In [1]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


In [2]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table


print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [3]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

In [4]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

## 1. ALF01.csv

In [5]:
df=load_data("QLF50.csv")

✓ Successfully loaded dataset with 5940 rows and 12 columns


,STATISTIC,Statistic Label,TLIST(Q1),Quarter,C02199V02655,Sex,C04283V05060,Education Attainment Level,C02076V02508,Age Group,UNIT,VALUE
0,QLF50C01,Employment rate,20191,2019Q1,-,Both sexes,-,Levels of Education (Levels 0-8),-,All ages,%,74.9
1,QLF50C01,Employment rate,20191,2019Q1,-,Both sexes,-,Levels of Education (Levels 0-8),365,20 - 24 years,%,64.0
2,QLF50C01,Employment rate,20191,2019Q1,-,Both sexes,-,Levels of Education (Levels 0-8),410,25 - 29 years,%,80.1
3,QLF50C01,Employment rate,20191,2019Q1,-,Both sexes,-,Levels of Education (Levels 0-8),4251,25 - 54 years,%,80.1
4,QLF50C01,Employment rate,20191,2019Q1,-,Both sexes,-,Levels of Education (Levels 0-8),440,30 - 34 years,%,81.4


In [6]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 5,940 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 12 columns:
  - Column names: ['STATISTIC', 'Statistic Label', 'TLIST(Q1)', 'Quarter', 'C02199V02655', 'Sex', 'C04283V05060', 'Education Attainment Level', 'C02076V02508', 'Age Group', 'UNIT', 'VALUE']


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5940 entries, 0 to 5939
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   STATISTIC                   5940 non-null   object 
 1   Statistic Label             5940 non-null   object 
 2   TLIST(Q1)                   5940 non-null   int64  
 3   Quarter                     5940 non-null   object 
 4   C02199V02655                5940 non-null   object 
 5   Sex                         5940 non-null   object 
 6   C04283V05060                5940 non-null   object 
 7   Education Attainment Level  5940 non-null   object 
 8   C02076V02508                5940 non-null   object 
 9   Age Group                   5940 non-null   object 
 10  UNIT                        5940 non-null   object 
 11  VALUE                       3968 non-null   float64
dtypes: float64(1), int64(1), object(10)
memory usage: 557.0+ KB


In [9]:


# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['TLIST(Q1)']=df['TLIST(Q1)'].astype('category')


# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 11


In [10]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,3968.0,74.122077,13.849446,26.9,66.0,76.7,85.0,97.7,33.198653,603


In [11]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
STATISTIC,5940,1,QLF50C01,5940,0.0,1
Statistic Label,5940,1,Employment rate,5940,0.0,1
TLIST(Q1),5940,30,20191,198,0.0,30
Quarter,5940,30,2019Q1,198,0.0,30
C02199V02655,5940,3,-,1980,0.0,3
Sex,5940,3,Both sexes,1980,0.0,3
C04283V05060,5940,6,-,990,0.0,6
Education Attainment Level,5940,6,Less than primary (Level 0),990,0.0,6
C02076V02508,5940,11,-,540,0.0,11
Age Group,5940,11,20 - 24 years,540,0.0,11


In [12]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


STATISTIC


,value,count
0,QLF50C01,5940



Statistic Label


,value,count
0,Employment rate,5940



TLIST(Q1)


...

,value,count
0,-,540
1,365,540
2,410,540
3,4251,540
4,440,540
5,460,540
6,475,540
7,495,540
8,515,540
9,530,540



Age Group


,value,count
0,20 - 24 years,540
1,25 - 29 years,540
2,25 - 54 years,540
3,30 - 34 years,540
4,35 - 39 years,540
5,40 - 44 years,540
6,45 - 49 years,540
7,50 - 54 years,540
8,55 - 59 years,540
9,60 - 64 years,540



UNIT


,value,count
0,%,5940


In [13]:
for col in category_columns:
    missing_stats = (
        df.groupby(col, observed=True, dropna=False)["VALUE"]
        .agg(
            total_rows="size",
            valid_rows="count",
        )
    )

    missing_stats["missing_rows"] = (
        missing_stats["total_rows"] - missing_stats["valid_rows"]
    )
    missing_stats["missing_pct"] = (
        missing_stats["missing_rows"]
        .div(missing_stats["total_rows"])
        .mul(100)
        .round(2)
    )

    print(f"\n{col}")
    display(missing_stats.sort_values("missing_pct", ascending=False))


STATISTIC


,total_rows,valid_rows,missing_rows,missing_pct
STATISTIC,,,,
QLF50C01,5940,3968,1972,33.2



Statistic Label


,total_rows,valid_rows,missing_rows,missing_pct
Statistic Label,,,,
Employment rate,5940,3968,1972,33.2



TLIST(Q1)


...

,total_rows,valid_rows,missing_rows,missing_pct
C02076V02508,,,,
365,540,273,267,49.44
410,540,283,257,47.59
440,540,303,237,43.89
460,540,351,189,35.00
550,540,361,179,33.15
530,540,370,170,31.48
515,540,374,166,30.74
475,540,375,165,30.56
495,540,378,162,30.00



Age Group


,total_rows,valid_rows,missing_rows,missing_pct
Age Group,,,,
20 - 24 years,540,273,267,49.44
25 - 29 years,540,283,257,47.59
30 - 34 years,540,303,237,43.89
35 - 39 years,540,351,189,35.00
60 - 64 years,540,361,179,33.15
55 - 59 years,540,370,170,31.48
50 - 54 years,540,374,166,30.74
40 - 44 years,540,375,165,30.56
45 - 49 years,540,378,162,30.00



UNIT


,total_rows,valid_rows,missing_rows,missing_pct
UNIT,,,,
%,5940,3968,1972,33.2
